
# Supervised Dialogue Act Classifier (BERT) + Inference on StudyChat

Train a **supervised Dialogue Act classifier** (BERT + softmax), then run inference on the `prompt` field of `wmcnicho/StudyChat`.
- Input format for training CSV(s): columns `text,label` (label is a SwDA code like `sd`, `b`, `qy`, ...).
- If you only have one CSV, the notebook creates a **stratified train/val split**.
- Outputs for StudyChat inference: JSON and CSV with `da_label`, `da_name`, `da_score`.


In [ ]:

# Optional: install deps if needed
# %pip install -q transformers datasets evaluate pandas scikit-learn python-dotenv torch tqdm


In [ ]:
!pip install -q evaluate
!pip install -U accelerate
!pip install -U transformers

In [ ]:
import transformers, torch
print("Transformers:", transformers.__version__)
print("Torch:", torch.__version__)

In [ ]:

import os, json, random, math
from dataclasses import dataclass
from typing import Dict, List
import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding
)
import evaluate
from dotenv import load_dotenv
from sklearn.model_selection import StratifiedShuffleSplit
import torch

os.environ["TRANSFORMERS_BACKEND"] = "pt"

print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())


In [ ]:

DA_CODE_TO_NAME = {
    "sd":"Statement-non-opinion","b":"Acknowledge (Backchannel)","sv":"Statement-opinion","%":"Uninterpretable",
    "aa":"Agree/Accept","ba":"Appreciation","qy":"Yes-No-Question","ny":"Yes Answers","fc":"Conventional-closing",
    "qw":"Wh-Question","nn":"No Answers","bk":"Response Acknowledgement","h":"Hedge","qy^d":"Declarative Yes-No-Question",
    "bh":"Backchannel in Question Form","^q":"Quotation","bf":"Summarize/Reformulate","fo_o_fw_\"_by_bc":"Other",
    "na":"Affirmative Non-yes Answers","ad":"Action-directive","^2":"Collaborative Completion","b^m":"Repeat-phrase",
    "qo":"Open-Question","qh":"Rhetorical-Question","^h":"Hold Before Answer/Agreement","ar":"Reject",
    "ng":"Negative Non-no Answers","br":"Signal-non-understanding","no":"Other Answers","fp":"Conventional-opening",
    "qrr":"Or-Clause","arp_nd":"Dispreferred Answers","t3":"3rd-party-talk","oo_co_cc":"Offers, Options, Commits",
    "aap_am":"Maybe/Accept-part","t1":"Downplayer","bd":"Self-talk","^g":"Tag-Question","qw^d":"Declarative Wh-Question",
    "fa":"Apology","ft":"Thanking", "cd":"software code", "ac": "Assignment Copy", "err": "software error"
}
# Extensions (optional):
# DA_CODE_TO_NAME.update({"eq":"Assignment-Statement","pc":"Programming-Code"})

LABELS = list(DA_CODE_TO_NAME.keys())
label2id = {lab:i for i, lab in enumerate(LABELS)}
id2label = {i:lab for lab,i in label2id.items()}
len(LABELS), LABELS[:10]


In [ ]:

# Config
MODEL_NAME = "bert-base-uncased"
MAX_LEN    = 128
BATCH      = 16
EPOCHS     = 3
LR         = 2e-5
WD         = 0.01
SEED       = 42

OUT_DIR    = "bert-da-swda"

TRAIN_CSV  = "swda_train.csv"   # expects text,label
VAL_CSV    = "swda_val.csv"     # expects text,label
SINGLE_CSV = None               # e.g., "swda_all.csv"
VAL_SIZE   = 0.1

STUDYCHAT_OUT_JSON = "studychat_supervised_da_with_new_labels.json"
STUDYCHAT_OUT_CSV  = "studychat_supervised_da_with_new_labels.csv"


In [ ]:

def df_to_dataset(df: pd.DataFrame) -> Dataset:
    df = df.dropna(subset=["text","label"]).copy()
    df = df[df["label"].isin(LABELS)]
    df["label_id"] = df["label"].map(label2id)
    return Dataset.from_pandas(df[["text","label","label_id"]], preserve_index=False)


In [ ]:

from sklearn.model_selection import StratifiedShuffleSplit
def make_stratified_split(single_csv: str, val_size: float = 0.1, seed: int = 42) -> DatasetDict:
    df = pd.read_csv(single_csv)
    if not {"text","label"}.issubset(df.columns):
        raise ValueError(f"`{single_csv}` must have columns text,label")
    df = df.dropna(subset=["text","label"])
    df = df[df["label"].isin(LABELS)].copy()
    y = df["label"].values
    sss = StratifiedShuffleSplit(n_splits=1, test_size=val_size, random_state=seed)
    (train_idx, val_idx), = sss.split(np.zeros(len(y)), y)
    df_tr = df.iloc[train_idx].reset_index(drop=True)
    df_va = df.iloc[val_idx].reset_index(drop=True)
    print("Split sizes:", len(df_tr), len(df_va))
    return DatasetDict(train=df_to_dataset(df_tr), validation=df_to_dataset(df_va))


In [ ]:

def load_supervised_data(train_csv: str = None, val_csv: str = None, single_csv: str = None, val_size: float = 0.1) -> DatasetDict:
    if single_csv:
        print("Using single CSV with stratified split:", single_csv)
        return make_stratified_split(single_csv, val_size=val_size, seed=SEED)
    if not train_csv or not val_csv:
        raise ValueError("Provide TRAIN_CSV and VAL_CSV, or set SINGLE_CSV.")
    print("Loading:", train_csv, "|", val_csv)
    df_tr = pd.read_csv(train_csv)
    df_va = pd.read_csv(val_csv)
    return DatasetDict(train=df_to_dataset(df_tr), validation=df_to_dataset(df_va))


In [ ]:

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, padding=False, max_length=128)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
metric_acc = evaluate.load("accuracy")
metric_f1  = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = metric_acc.compute(predictions=preds, references=labels)["accuracy"]
    f1m = metric_f1.compute(predictions=preds, references=labels, average="macro")["f1"]
    return {"accuracy": acc, "macro_f1": f1m}


In [ ]:
def train_supervised(dsd: DatasetDict, out_dir: str):
    # cria 'labels' a partir de 'label_id' e tokeniza
    def preprocess(batch):
        enc = tokenizer(
            batch["text"],
            truncation=True,
            padding=False,
            max_length=MAX_LEN,
        )
        enc["labels"] = batch["label_id"]   # <= o Trainer espera 'labels'
        return enc

    dsd_tok = dsd.map(
        preprocess,
        batched=True,
        remove_columns=["text", "label", "label_id"],  # já que criamos 'labels'
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(LABELS),
        id2label={i: id2label[i] for i in range(len(LABELS))},
        label2id={id2label[i]: i for i in range(len(LABELS))}
    )

    args = TrainingArguments(
        output_dir=out_dir,
        learning_rate=LR,
        per_device_train_batch_size=BATCH,
        per_device_eval_batch_size=BATCH,
        num_train_epochs=EPOCHS,
        weight_decay=WD,
        eval_strategy="epoch",        # você já corrigiu p/ versões novas
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        logging_steps=50,
        seed=SEED,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=dsd_tok["train"],
        eval_dataset=dsd_tok["validation"],
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )
    trainer.train()
    trainer.save_model(out_dir)
    tokenizer.save_pretrained(out_dir)
    print("Best model saved to:", out_dir)
    return out_dir


In [ ]:

from dataclasses import dataclass

@dataclass
class InferConfig:
    model_dir: str
    out_json: str = "studychat_supervised_da_with_new_labels.json"
    out_csv:  str = "studychat_supervised_da_with_new_labels.csv"
    max_len:  int = 128
    batch:    int = 64

def infer_studychat(cfg: InferConfig):
    load_dotenv('.env')
    HF_TOKEN = os.environ.get("HF_TOKEN")
    if not HF_TOKEN:
        print("⚠️ HF_TOKEN not set; gated dataset access may fail.")
    ds = load_dataset("wmcnicho/StudyChat", split="train", token=HF_TOKEN)
    print(ds)

    model = AutoModelForSequenceClassification.from_pretrained(cfg.model_dir)
    tok = AutoTokenizer.from_pretrained(cfg.model_dir, use_fast=True)

    texts = [
        str(x) if x is not None else ""
        for col in ["prompt", "response"]
        for x in ds[col]
    ]

    def _chunks(lst, n):
        for i in range(0, len(lst), n):
            yield lst[i:i+n]

    def _tok_batch(text_list):
        return tok(text_list, truncation=True, padding=True, max_length=cfg.max_len, return_tensors="pt")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.eval()

    pred_codes, pred_names, pred_scores = [], [], []
    with torch.no_grad():
        for batch in _chunks(texts, cfg.batch):
            enc = _tok_batch(batch).to(device)
            out = model(**enc)
            probs = torch.nn.functional.softmax(out.logits, dim=-1).cpu().numpy()
            idxs = probs.argmax(axis=1)
            scores = probs.max(axis=1)
            for i, s in zip(idxs, scores):
                code = id2label[int(i)]
                name = DA_CODE_TO_NAME[code]
                pred_codes.append(code)
                pred_names.append(name)
                pred_scores.append(float(s))

    df = ds.to_pandas()
    df["da_label"] = pred_codes
    df["da_name"]  = pred_names
    df["da_score"] = pred_scores
    df.to_csv(cfg.out_csv, index=False)
    df.to_json(cfg.out_json, orient="records", force_ascii=False, indent=2)
    print("Saved:", cfg.out_csv, "and", cfg.out_json)



## Quick start

- Put your labeled CSVs (`text,label`) in the working directory (or set `SINGLE_CSV`).
- Run **TRAIN** then **INFER** cells below.


In [ ]:

# === TRAIN ===
random.seed(42); np.random.seed(42); torch.manual_seed(42)
from datasets import DatasetDict
dsd = None
if (SINGLE_CSV is not None) and isinstance(SINGLE_CSV, str):
    dsd = make_stratified_split(SINGLE_CSV, val_size=VAL_SIZE, seed=SEED)
else:
    dsd = load_supervised_data(TRAIN_CSV, VAL_CSV, None, VAL_SIZE)
model_dir = train_supervised(dsd, out_dir=OUT_DIR)


In [ ]:

# === INFER on StudyChat (prompt) ===
infer_studychat(InferConfig(
    model_dir=OUT_DIR,
    out_json=STUDYCHAT_OUT_JSON,
    out_csv=STUDYCHAT_OUT_CSV
))
